# Store Sales Forecasting — model comparison & RMSE optimisation

Self-contained prototype that compares four models (LightGBM, XGBoost,
Random Forest, Ridge) on the weekly-sales forecasting task, with a heavy
emphasis on feature engineering.

**Sections**

1. Load the data
2. Quick EDA
3. Basic feature engineering (calendar + holiday flags + cyclical encoding + trend)
4. Advanced feature engineering (lags, rolling stats)
5. Time-based train / val / test split
6. Train four models
7. Test-set comparison
8. Ablation — do the lag/rolling features actually help?
9. Simple ensemble
10. Predictions vs. actuals (best model)
11. Save best artifact

**Why these choices**

- **Lag and rolling features** are the single biggest lever for RMSE on
  weekly retail sales. They represent a 1-step-ahead forecasting setup
  (last week's actual is available when predicting next week's sales).
- **Cyclical (sin/cos) encoding** of week-of-year and month gives linear
  models a way to learn seasonality without one-hot bloat.
- `store` is treated natively as categorical for LightGBM, but one-hot
  encoded for Random Forest and Ridge (where integer store IDs would be
  meaningless as a continuous feature).

In [44]:
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Locate the data file whether Jupyter was launched from project root or notebooks/.
HERE = Path.cwd()
ROOT = next(
    (c for c in (HERE, HERE.parent) if (c / "data" / "stores-sales.csv").exists()),
    HERE,
)
DATA_PATH = ROOT / "data" / "stores-sales.csv"
MODEL_PATH = ROOT / "models" / "model.pkl"

print(f"Data:  {DATA_PATH}  ({'exists' if DATA_PATH.exists() else 'MISSING'})")
print(f"Model: {MODEL_PATH}")

Data:  c:\Users\pc\store-sales-forecasting\data\stores-sales.csv  (exists)
Model: c:\Users\pc\store-sales-forecasting\models\model.pkl


## 1. Load the data

In [45]:
raw = pd.read_csv(DATA_PATH)
print(f"Shape: {raw.shape}")
raw.head()

Shape: (6435, 8)


,store,date,weekly_sales,holiday_flag,temperature,fuel_Price,cpi,unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [46]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   store         6435 non-null   int64  
 1   date          6435 non-null   object 
 2   weekly_sales  6435 non-null   float64
 3   holiday_flag  6435 non-null   int64  
 4   temperature   6435 non-null   float64
 5   fuel_Price    6435 non-null   float64
 6   cpi           6435 non-null   float64
 7   unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [47]:
raw.describe()

,store,weekly_sales,holiday_flag,temperature,fuel_Price,cpi,unemployment
count,6435.000000,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
std,12.988182,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885
min,1.000000,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000


## 2. Quick EDA

Total weekly sales over time, store size variation, and the holiday effect.

In [48]:
plot_df = raw.copy()
plot_df["date"] = pd.to_datetime(plot_df["date"], dayfirst=True)

totals = plot_df.groupby("date", as_index=False)["weekly_sales"].sum()
fig = px.line(totals, x="date", y="weekly_sales",
              title="Total weekly sales across all stores")
fig.show()

In [49]:
by_store = (
    plot_df.groupby("store")["weekly_sales"].mean()
    .sort_values()
    .reset_index()
)
fig = px.bar(by_store, x="store", y="weekly_sales",
             title="Mean weekly sales by store")
fig.update_layout(xaxis_type="category")
fig.show()

In [50]:
holiday_summary = (
    plot_df.groupby("holiday_flag")["weekly_sales"]
    .agg(["mean", "median", "count"])
    .rename(index={0: "non-holiday week", 1: "holiday week"})
)
holiday_summary

,mean,median,count
holiday_flag,,,
non-holiday week,1.041256e+06,956211.20,5985
holiday week,1.122888e+06,1018538.04,450


## 3. Basic feature engineering

Calendar parts, holiday flags, **cyclical encoding** (sin/cos of week and
month so models can see seasonality as a smooth periodic signal rather than
a categorical one), a **linear time trend**, and **pre-holiday flags** for
the week before each labelled holiday (the build-up week often shows a sales
spike of its own).

In [51]:
SUPER_BOWL_DATES = pd.to_datetime(
    ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"]
)
LABOUR_DAY_DATES = pd.to_datetime(
    ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"]
)
THANKSGIVING_DATES = pd.to_datetime(
    ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"]
)
CHRISTMAS_DATES = pd.to_datetime(
    ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"]
)
WEEK = pd.Timedelta(days=7)
PRE_SUPER_BOWL_DATES = SUPER_BOWL_DATES - WEEK
PRE_LABOUR_DAY_DATES = LABOUR_DAY_DATES - WEEK
PRE_THANKSGIVING_DATES = THANKSGIVING_DATES - WEEK
PRE_CHRISTMAS_DATES = CHRISTMAS_DATES - WEEK

In [52]:
def build_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    """Calendar + holiday + cyclical + trend features. Idempotent."""
    out = df.copy()

    # Source CSV uses DD-MM-YYYY; without dayfirst, "05-02-2010" parses as May 2.
    out["date"] = pd.to_datetime(out["date"], dayfirst=True)
    if "fuel_Price" in out.columns:
        out = out.rename(columns={"fuel_Price": "fuel_price"})
    out["store"] = out["store"].astype(int)

    out["year"] = out["date"].dt.year.astype(int)
    out["month"] = out["date"].dt.month.astype(int)
    out["week_of_year"] = out["date"].dt.isocalendar().week.astype(int)
    out["quarter"] = out["date"].dt.quarter.astype(int)
    out["day_of_year"] = out["date"].dt.dayofyear.astype(int)

    # Cyclical encoding — smoothly periodic, no jump from week 52 -> 1.
    out["week_sin"] = np.sin(2 * np.pi * out["week_of_year"] / 52)
    out["week_cos"] = np.cos(2 * np.pi * out["week_of_year"] / 52)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)

    # Linear time trend across the whole dataset.
    out["days_since_start"] = (out["date"] - out["date"].min()).dt.days

    # Holiday-week flags.
    out["is_super_bowl"] = out["date"].isin(SUPER_BOWL_DATES).astype(int)
    out["is_labour_day"] = out["date"].isin(LABOUR_DAY_DATES).astype(int)
    out["is_thanksgiving"] = out["date"].isin(THANKSGIVING_DATES).astype(int)
    out["is_christmas"] = out["date"].isin(CHRISTMAS_DATES).astype(int)

    # Pre-holiday flags — the week before each holiday often shows a build-up.
    out["is_pre_super_bowl"] = out["date"].isin(PRE_SUPER_BOWL_DATES).astype(int)
    out["is_pre_labour_day"] = out["date"].isin(PRE_LABOUR_DAY_DATES).astype(int)
    out["is_pre_thanksgiving"] = out["date"].isin(PRE_THANKSGIVING_DATES).astype(int)
    out["is_pre_christmas"] = out["date"].isin(PRE_CHRISTMAS_DATES).astype(int)

    return out


features_df = build_basic_features(raw)
new_cols = sorted(set(features_df.columns) - set(raw.columns) - {"fuel_price"})
print(f"Engineered ({len(new_cols)}): {new_cols}")
features_df.head()

Engineered (18): ['day_of_year', 'days_since_start', 'is_christmas', 'is_labour_day', 'is_pre_christmas', 'is_pre_labour_day', 'is_pre_super_bowl', 'is_pre_thanksgiving', 'is_super_bowl', 'is_thanksgiving', 'month', 'month_cos', 'month_sin', 'quarter', 'week_cos', 'week_of_year', 'week_sin', 'year']


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,...,month_cos,days_since_start,is_super_bowl,is_labour_day,is_thanksgiving,is_christmas,is_pre_super_bowl,is_pre_labour_day,is_pre_thanksgiving,is_pre_christmas
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,...,5.000000e-01,0,0,0,0,0,1,0,0,0
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,...,5.000000e-01,7,1,0,0,0,0,0,0,0
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,...,5.000000e-01,14,0,0,0,0,0,0,0,0
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,...,5.000000e-01,21,0,0,0,0,0,0,0,0
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,...,6.123234e-17,28,0,0,0,0,0,0,0,0


## 4. Advanced feature engineering — lag & rolling stats

The strongest predictors of next week's sales are last week's sales. We add
**lag features** (1, 2, and 4 weeks back) and **rolling statistics** (mean
and std over a 4- and 12-week trailing window, computed with `shift(1)` so
they only see history strictly before the current row).

Each rolling/lag column has NaNs for the first few weeks of each store's
history. We drop those rows once features are built; this represents a
practical 1-step-ahead forecasting setup where last week's actual is
available when predicting next week.

In [53]:
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per-store lag and rolling stats on weekly_sales.

    Requires `store`, `date`, and `weekly_sales` columns. Sorts by
    (store, date) so the within-group shifts/rollings are time-aligned.
    """
    out = df.sort_values(["store", "date"]).copy()
    g = out.groupby("store")["weekly_sales"]

    for lag in (1, 2, 4):
        out[f"lag_{lag}_sales"] = g.transform(lambda s, k=lag: s.shift(k))

    # shift(1) before the rolling so the window doesn't include the current row.
    for win in (4, 12):
        out[f"rolling_mean_{win}"] = g.transform(
            lambda s, w=win: s.shift(1).rolling(w).mean()
        )
    out["rolling_std_4"] = g.transform(lambda s: s.shift(1).rolling(4).std())

    return out.reset_index(drop=True)


features_full = add_lag_features(features_df)

LAG_COLS = [
    "lag_1_sales", "lag_2_sales", "lag_4_sales",
    "rolling_mean_4", "rolling_mean_12", "rolling_std_4",
]
print(f"NaN rows by lag column:")
print(features_full[LAG_COLS].isna().sum())

NaN rows by lag column:
lag_1_sales         45
lag_2_sales         90
lag_4_sales        180
rolling_mean_4     180
rolling_mean_12    540
rolling_std_4      180
dtype: int64


In [54]:
# Drop the per-store warm-up rows that lack a full rolling-12 history.
before = len(features_full)
features_full = features_full.dropna(subset=LAG_COLS).reset_index(drop=True)
print(f"Dropped {before - len(features_full)} warm-up rows; {len(features_full)} remain")

NUMERIC_FEATURES = [
    "holiday_flag", "temperature", "fuel_price", "cpi", "unemployment",
    "year", "month", "week_of_year", "quarter", "day_of_year",
    "week_sin", "week_cos", "month_sin", "month_cos", "days_since_start",
    "is_super_bowl", "is_labour_day", "is_thanksgiving", "is_christmas",
    "is_pre_super_bowl", "is_pre_labour_day", "is_pre_thanksgiving",
    "is_pre_christmas",
]
FEATURE_COLUMNS = ["store"] + NUMERIC_FEATURES + LAG_COLS
CATEGORICAL_FEATURES = ["store"]
TARGET = "weekly_sales"

print(f"Total features: {len(FEATURE_COLUMNS)}")

Dropped 540 warm-up rows; 5895 remain
Total features: 30


## 5. Time-based train / val / test split

Strictly chronological. The test period (`>= 2012-06-01`) is held out; the
last ~8 weeks of train act as inner validation for early stopping.

In [55]:
def time_based_split(df, cutoff):
    cutoff = pd.Timestamp(cutoff)
    return df[df["date"] < cutoff].copy(), df[df["date"] >= cutoff].copy()


TRAIN_TEST_CUTOFF = "2012-06-01"
TRAIN_VAL_CUTOFF = "2012-04-01"

train_full, test = time_based_split(features_full, TRAIN_TEST_CUTOFF)
train_inner, val = time_based_split(train_full, TRAIN_VAL_CUTOFF)


def _r(d):
    return f"{d['date'].min().date()} \u2192 {d['date'].max().date()}"


print(f"train (full): {len(train_full):>5}  rows  {_r(train_full)}")
print(f"  inner:      {len(train_inner):>5}  rows  {_r(train_inner)}")
print(f"  val:        {len(val):>5}  rows  {_r(val)}")
print(f"test:         {len(test):>5}  rows  {_r(test)}")

X_train_full = train_full[FEATURE_COLUMNS]; y_train_full = train_full[TARGET]
X_inner = train_inner[FEATURE_COLUMNS];     y_inner = train_inner[TARGET]
X_val = val[FEATURE_COLUMNS];                y_val = val[TARGET]
X_test = test[FEATURE_COLUMNS];              y_test = test[TARGET]

train (full):  4905  rows  2010-04-30 → 2012-05-25
  inner:       4545  rows  2010-04-30 → 2012-03-30
  val:          360  rows  2012-04-06 → 2012-05-25
test:           990  rows  2012-06-01 → 2012-10-26


## 6. Train four models

A common evaluation harness, then four models with their per-flavour
preprocessing. Tree boosters (LightGBM, XGBoost) take `store` as an integer
feature; Random Forest and Ridge get a one-hot encoding because integer
store IDs are not meaningfully ordinal for them.

In [56]:
def compute_metrics(y_true, y_pred) -> dict:
    return {
        "rmse": float(root_mean_squared_error(y_true, y_pred)),
        "mae":  float(mean_absolute_error(y_true, y_pred)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
    }


def one_hot_store(X: pd.DataFrame) -> pd.DataFrame:
    """Replace `store` (int id) with a 0/1 one-hot block. Used by RF and Ridge."""
    return pd.get_dummies(X, columns=["store"], prefix="store").astype(float)


X_inner_oh = one_hot_store(X_inner)
X_val_oh = one_hot_store(X_val)
X_train_full_oh = one_hot_store(X_train_full)
# Reindex test one-hot to match the training columns in case any store is missing.
X_test_oh = one_hot_store(X_test).reindex(columns=X_train_full_oh.columns, fill_value=0.0)
print(f"One-hot feature count: {X_train_full_oh.shape[1]}")

One-hot feature count: 74


### 6.1 LightGBM

In [57]:
LGBM_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    random_state=42,
    verbose=-1,
)

lgbm = lgb.LGBMRegressor(**LGBM_PARAMS)
lgbm.fit(
    X_inner, y_inner,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    categorical_feature=CATEGORICAL_FEATURES,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)
print(f"LightGBM best_iteration: {lgbm.best_iteration_}")

LightGBM best_iteration: 266


### 6.2 XGBoost

In [58]:
XGB_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    early_stopping_rounds=50,
    random_state=42,
    verbosity=0,
    n_jobs=-1,
    tree_method="hist",
)

xgb_model = xgb.XGBRegressor(**XGB_PARAMS)
xgb_model.fit(
    X_inner, y_inner,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"XGBoost best_iteration: {xgb_model.best_iteration}")

XGBoost best_iteration: 223


### 6.3 Random Forest (with one-hot store)

In [59]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_leaf=2,
    max_features=0.6,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_full_oh, y_train_full)
print(f"Random Forest fitted: {rf.n_estimators} trees, depth\u2264{rf.max_depth}")

Random Forest fitted: 400 trees, depth≤20


### 6.4 Ridge regression (linear baseline)

A scaled linear model is the no-tree baseline. Without lag features it would
be hopeless on this problem; with them it actually becomes competitive.

In [60]:
ridge = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=1.0, random_state=42)),
])
ridge.fit(X_train_full_oh, y_train_full)
print("Ridge regression fitted.")

Ridge regression fitted.


## 7. Test-set comparison

In [61]:
preds = {
    "LightGBM":     lgbm.predict(X_test),
    "XGBoost":      xgb_model.predict(X_test),
    "RandomForest": rf.predict(X_test_oh),
    "Ridge":        ridge.predict(X_test_oh),
}
test_metrics = pd.DataFrame(
    {name: compute_metrics(y_test, p) for name, p in preds.items()}
).T.sort_values("rmse")
test_metrics

,rmse,mae,mape
LightGBM,53378.034117,35953.135310,0.035143
XGBoost,54837.410994,36980.279826,0.036169
RandomForest,60064.857816,41873.853521,0.040850
Ridge,66799.043973,48492.143030,0.052103


In [62]:
fig = px.bar(
    test_metrics.reset_index().rename(columns={"index": "model"}),
    x="model", y="rmse", text="rmse",
    title="Test RMSE by model (lower is better)",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=420)
fig.show()

## 8. Ablation — do the lag and rolling features actually help?

Same train/test split, same LightGBM hyperparameters, only the feature set
differs. The "basic" arm uses calendar + holiday + cyclical + trend
features; the "full" arm adds lag and rolling features on top. The
answer should explain most of the RMSE we're claiming.

In [63]:
BASIC_FEATURE_COLUMNS = ["store"] + NUMERIC_FEATURES  # no LAG_COLS
print(f"Basic features ({len(BASIC_FEATURE_COLUMNS)}): {BASIC_FEATURE_COLUMNS}")
print(f"Added by LAG_COLS ({len(LAG_COLS)}): {LAG_COLS}")

Basic features (24): ['store', 'holiday_flag', 'temperature', 'fuel_price', 'cpi', 'unemployment', 'year', 'month', 'week_of_year', 'quarter', 'day_of_year', 'week_sin', 'week_cos', 'month_sin', 'month_cos', 'days_since_start', 'is_super_bowl', 'is_labour_day', 'is_thanksgiving', 'is_christmas', 'is_pre_super_bowl', 'is_pre_labour_day', 'is_pre_thanksgiving', 'is_pre_christmas']
Added by LAG_COLS (6): ['lag_1_sales', 'lag_2_sales', 'lag_4_sales', 'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4']


In [64]:
# Same LightGBM hyperparameters as section 6.1, just a different X.
lgbm_basic = lgb.LGBMRegressor(**LGBM_PARAMS)
lgbm_basic.fit(
    train_inner[BASIC_FEATURE_COLUMNS], y_inner,
    eval_set=[(val[BASIC_FEATURE_COLUMNS], y_val)],
    eval_metric="rmse",
    categorical_feature=CATEGORICAL_FEATURES,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)
print(f"Basic-features best_iteration: {lgbm_basic.best_iteration_}")

Basic-features best_iteration: 348


In [65]:
basic_metrics = compute_metrics(y_test, lgbm_basic.predict(test[BASIC_FEATURE_COLUMNS]))
full_metrics  = compute_metrics(y_test, lgbm.predict(X_test))

ablation = pd.DataFrame({
    "basic (no lag/rolling)":   basic_metrics,
    "full (with lag/rolling)":  full_metrics,
})
ablation["absolute_improvement"] = ablation["basic (no lag/rolling)"] - ablation["full (with lag/rolling)"]
ablation["pct_improvement"]      = ablation["absolute_improvement"] / ablation["basic (no lag/rolling)"]
ablation

,basic (no lag/rolling),full (with lag/rolling),absolute_improvement,pct_improvement
rmse,92846.965107,53378.034117,39468.930990,0.425097
mae,63972.983620,35953.135310,28019.848310,0.437995
mape,0.074219,0.035143,0.039075,0.526487


In [66]:
viz = pd.DataFrame({
    "configuration": ["basic (no lag/rolling)", "full (with lag/rolling)"],
    "test_rmse":     [basic_metrics["rmse"], full_metrics["rmse"]],
})
fig = px.bar(viz, x="configuration", y="test_rmse", text="test_rmse",
             title="Test RMSE — basic features vs. with lag/rolling")
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=420, showlegend=False)
fig.show()

## 9. Simple ensemble

Equal-weight average of the top tree models. Ensembles trade a small amount
of complexity for a typical 1\u20135% RMSE drop because the models' errors
are partially uncorrelated.

In [67]:
ranked = list(test_metrics.index)
top3 = ranked[:3]
print(f"Averaging top-3 models: {top3}")

ensemble_pred = np.mean([preds[m] for m in top3], axis=0)
ensemble_metrics = compute_metrics(y_test, ensemble_pred)

with_ensemble = test_metrics.copy()
with_ensemble.loc[f"Ensemble({'+'.join(top3)})"] = ensemble_metrics
with_ensemble.sort_values("rmse")

Averaging top-3 models: ['LightGBM', 'XGBoost', 'RandomForest']


,rmse,mae,mape
LightGBM,53378.034117,35953.135310,0.035143
Ensemble(LightGBM+XGBoost+RandomForest),54365.580429,36898.956139,0.035961
XGBoost,54837.410994,36980.279826,0.036169
RandomForest,60064.857816,41873.853521,0.040850
Ridge,66799.043973,48492.143030,0.052103


## 10. Best model — predictions vs. actuals

In [68]:
best_name = test_metrics["rmse"].idxmin()
best_pred = preds[best_name]
print(f"Best single model: {best_name}  (RMSE = {test_metrics.loc[best_name, 'rmse']:,.2f})")

test_results = test[["store", "date", "weekly_sales"]].copy()
test_results["predicted"] = best_pred

STORE = 1  # change to inspect another store
ts = test_results[test_results["store"] == STORE].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts["date"], y=ts["weekly_sales"],
                         mode="lines+markers", name="Actual"))
fig.add_trace(go.Scatter(x=ts["date"], y=ts["predicted"],
                         mode="lines+markers", name=f"Predicted ({best_name})"))
fig.update_layout(title=f"Store {STORE} \u2014 actual vs. predicted (test period)",
                  xaxis_title="date", yaxis_title="weekly_sales",
                  hovermode="x unified")
fig.show()

Best single model: LightGBM  (RMSE = 53,378.03)


In [69]:
residuals = y_test.values - best_pred
fig = px.histogram(residuals, nbins=40,
                   title=f"Test residuals \u2014 {best_name} (actual \u2212 predicted)")
fig.update_layout(showlegend=False, xaxis_title="residual", yaxis_title="count")
fig.show()

## 11. Save artifact

Save the best single model with the advanced feature set to
`models/model.pkl`. The schema matches what `src/train.py` writes — the
Streamlit app picks it up from there.

In [70]:
if best_name == "LightGBM":
    best_model = lgbm
elif best_name == "XGBoost":
    best_model = xgb_model
elif best_name == "RandomForest":
    best_model = rf
else:
    best_model = ridge

uses_one_hot = best_name in {"RandomForest", "Ridge"}

# Per-store fallback stats — used by the Streamlit app to fill NaN lag
# values for uploads that don't carry full per-store history.
store_stats = (
    train_full.groupby("store")["weekly_sales"].agg(["mean", "std"])
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
artifact = {
    "model": best_model,
    "model_name": best_name,
    "feature_columns": FEATURE_COLUMNS if not uses_one_hot else list(X_train_full_oh.columns),
    "categorical_features": CATEGORICAL_FEATURES,
    "lag_columns": LAG_COLS,
    "store_stats": store_stats,
    "uses_one_hot_store": uses_one_hot,
    "train_metrics": compute_metrics(y_train_full, (
        best_model.predict(X_train_full_oh) if uses_one_hot
        else best_model.predict(X_train_full)
    )),
    "test_metrics": compute_metrics(y_test, best_pred),
    "ensemble_test_metrics": ensemble_metrics,
    "ensemble_members": top3,
    "train_test_cutoff": TRAIN_TEST_CUTOFF,
    "n_train_rows": int(len(train_full)),
    "n_test_rows":  int(len(test)),
}
joblib.dump(artifact, MODEL_PATH)
print(f"Saved best model ({best_name}) to {MODEL_PATH}")
print(f"  test RMSE = {artifact['test_metrics']['rmse']:,.2f}")
print(f"  ensemble  = {ensemble_metrics['rmse']:,.2f}")

Saved best model (LightGBM) to c:\Users\pc\store-sales-forecasting\models\model.pkl
  test RMSE = 53,378.03
  ensemble  = 54,365.58
